# **NavDrishti: Indian Sign Language AI Translation System**
### *Capturing Non-manual Features of Indian Sign Language and Converting Them into Text*

This notebook trains a complete, end-to-end, working temporal sequence recognition model on the **ISL_CSLRT_Corpus** dataset (~8 GB, hosted on Kaggle). The training uses a **BiLSTM + Attention** model over MediaPipe Holistic facial and pose features.

---

### **1. GPU check**
Check if a GPU runtime is enabled. This will accelerate training of the BiLSTM model.

In [ ]:
!nvidia-smi
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Active GPU Device:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    print("GPU not found. Running on CPU.")
    device = torch.device("cpu")

### **2. Dependency installation**
Install the required libraries for feature extraction (MediaPipe, OpenCV) and evaluation.

In [ ]:
!pip install opencv-python mediapipe pandas numpy openpyxl gdown scikit-learn matplotlib seaborn tqdm

### **3. Kaggle authentication**
Provide credentials to download the dataset from Kaggle. Select your `kaggle.json` file when prompted.

In [ ]:
from google.colab import files
import os

if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Please upload your 'kaggle.json' token file downloaded from Kaggle Account Settings:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials set up successfully!")
else:
    print("Kaggle token already exists.")

### **4. Dataset download**
Download the dataset (`kartiksaxena/isl-csltr`) via the Kaggle API.

In [ ]:
!kaggle datasets download -d kartiksaxena/isl-csltr

### **5. Dataset extraction**
Extract the downloaded 8 GB zip archive into `/content/ISL_CSLRT_Corpus`.

In [ ]:
import os
import zipfile

zip_path = "/content/isl-csltr.zip"
extract_path = "/content/ISL_CSLRT_Corpus"

if not os.path.exists(extract_path):
    print("Extracting dataset (this may take a few minutes)...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete!")
else:
    print("Dataset already extracted.")

### **6. Dataset inspection**
Create the `config.py` file dynamically in Colab and inspect the dataset's files, classes, and resolutions.

In [ ]:
# Write config.py
colab_config = """
import os
DATASET_PATH = "/content/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus"
FEATURES_DIR = "/content/features"
MODELS_DIR = "/content/models"

EXCEL_DETAILS_PATH = os.path.join(DATASET_PATH, "corpus_csv_files", "ISL_CSLRT_Corpus details.xlsx")
GLOSS_CSV_PATH = os.path.join(DATASET_PATH, "corpus_csv_files", "ISL Corpus sign glosses.csv")

os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

SEQUENCE_LENGTH = 30
INPUT_DIM = 534
HIDDEN_DIM = 128
LSTM_LAYERS = 2
DROPOUT = 0.3
BATCH_SIZE = 16
EPOCHS = 35
LEARNING_RATE = 1e-3
CONFIDENCE_THRESHOLD = 0.60
STABILITY_THRESHOLD = 4
SMOOTHING_WINDOW = 8

SELECTED_FACE_INDICES = [
    70, 63, 105, 66, 107, 55, 65, 52, 53, 46,
    300, 293, 334, 296, 336, 285, 295, 282, 283, 276,
    33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246,
    362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398,
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 95,
    78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308, 415, 310, 311, 312, 13, 82, 81, 80, 191,
    1, 2, 98, 327, 4, 5, 6, 197, 195, 168,
    0, 17, 18, 200, 152, 377, 400, 396, 175, 171, 148, 136, 150, 176, 140, 142
]
POSE_INDICES = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
"""
with open("config.py", "w") as f:
    f.write(colab_config)
print("config.py created!")

In [ ]:
# Run inspection statistics
import pandas as pd
import os
import config

df = pd.read_excel(config.EXCEL_DETAILS_PATH)
df = df.dropna(subset=['Sentences'])
df = df[df['File location'].apply(lambda x: isinstance(x, str) and not x.startswith('>>>'))]

print("=== DATASET SUMMARY ===")
print(f"Total videos in mapping sheet: {len(df)}")
print(f"Total unique sentence classes: {df['Sentences'].nunique()}")
print("\nFirst 5 rows of Excel details:")
print(df.head())

print("\nClass Distribution (Top 10 Classes):")
print(df['Sentences'].value_counts().head(10))

### **7. Dataset visualization**
Decodes a sample video file from the dataset and displays its properties (resolution, FPS, frame counts).

In [ ]:
import cv2
import random
import os

sample_row = df.sample(1).iloc[0]
video_rel_path = sample_row['File location']
video_full_path = os.path.join(os.path.dirname(config.DATASET_PATH), video_rel_path)

cap = cv2.VideoCapture(video_full_path)
if cap.isOpened():
    w = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    print(f"Sample Video: {video_rel_path}")
    print(f"Resolution: {int(w)}x{int(h)}")
    print(f"Frame Rate (FPS): {fps:.2f}")
    print(f"Total Frames: {frames}")
    print(f"Duration: {frames/fps:.2f} seconds")
else:
    print(f"Could not open video {video_full_path}")
cap.release()

### **8. Data preprocessing**
Prepares the framework to sample and align sequences to exactly 30 frames. Resumable feature extraction checks if `.npy` exists for a file and skips if done.

### **9. Feature extraction**
Implements the MediaPipe Holistic feature extractor, capturing facial expressions, eyebrows, head pose, hand gesture coordinates, and pose landmarks. Runs the extraction in batches and outputs `.npy` files.

In [ ]:
# Create directories for script files
!mkdir -p ml

# Write feature_extraction.py
with open("ml/feature_extraction.py", "w") as f:
    f.write('''
import cv2
import mediapipe as mp
import numpy as np
import math
import config

class FeatureExtractor:
    def __init__(self, static_image_mode=False):
        self.mp_holistic = mp.solutions.holistic
        self.holistic = self.mp_holistic.Holistic(
            static_image_mode=static_image_mode,
            model_complexity=1,
            refine_face_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.model_points_3d = np.array([
            (0.0, 0.0, 0.0),
            (0.0, -330.0, -65.0),
            (-225.0, 170.0, -135.0),
            (225.0, 170.0, -135.0),
            (-150.0, -150.0, -125.0),
            (150.0, -150.0, -125.0)
        ], dtype=np.float32)

    def close(self):
        self.holistic.close()

    def _calculate_ear(self, landmarks, eye_indices):
        p1 = np.array(landmarks[eye_indices[0]])
        p2 = np.array(landmarks[eye_indices[4]])
        p3 = np.array(landmarks[eye_indices[5]])
        p4 = np.array(landmarks[eye_indices[8]])
        p5 = np.array(landmarks[eye_indices[12]])
        p6 = np.array(landmarks[eye_indices[13]])
        d_vert1 = np.linalg.norm(p2 - p6)
        d_vert2 = np.linalg.norm(p3 - p5)
        d_horiz = np.linalg.norm(p1 - p4)
        if d_horiz < 1e-6: return 0.0
        return (d_vert1 + d_vert2) / (2.0 * d_horiz)

    def _estimate_head_pose(self, landmarks_3d, width, height):
        pts_idx = [1, 152, 33, 263, 61, 291]
        image_points = np.array([
            [landmarks_3d[i][0] * width, landmarks_3d[i][1] * height] for i in pts_idx
        ], dtype=np.float32)
        focal_length = width
        center = (width / 2.0, height / 2.0)
        camera_matrix = np.array([
            [focal_length, 0, center[0]],
            [0, focal_length, center[1]],
            [0, 0, 1]
        ], dtype=np.float32)
        dist_coeffs = np.zeros((4, 1))
        success, rvec, tvec = cv2.solvePnP(self.model_points_3d, image_points, camera_matrix, dist_coeffs)
        if not success: return 0.0, 0.0, 0.0
        rmat, _ = cv2.Rodrigues(rvec)
        sy = math.sqrt(rmat[0, 0] * rmat[0, 0] + rmat[1, 0] * rmat[1, 0])
        if sy > 1e-6:
            pitch = math.atan2(rmat[2, 1], rmat[2, 2])
            yaw = math.atan2(-rmat[2, 0], sy)
            roll = math.atan2(rmat[1, 0], rmat[0, 0])
        else:
            pitch = math.atan2(-rmat[1, 2], rmat[1, 1])
            yaw = math.atan2(-rmat[2, 0], sy)
            roll = 0.0
        return math.degrees(pitch), math.degrees(yaw), math.degrees(roll)

    def extract_features(self, frame):
        height, width, _ = frame.shape
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.holistic.process(frame_rgb)
        
        features_flat = np.zeros(config.INPUT_DIM, dtype=np.float32)
        cues = {}
        
        if not results.face_landmarks:
            return features_flat, cues
            
        face_lms = np.array([[lm.x, lm.y, lm.z] for lm in results.face_landmarks.landmark])
        nose_tip = face_lms[1]
        centered_face = face_lms - nose_tip
        inter_ocular_dist = np.linalg.norm(face_lms[33] - face_lms[263])
        normalized_face = centered_face / (inter_ocular_dist + 1e-6)
        
        left_eye_idx = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
        right_eye_idx = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
        
        ear_l = self._calculate_ear(face_lms, left_eye_idx)
        ear_r = self._calculate_ear(face_lms, right_eye_idx)
        ear_avg = (ear_l + ear_r) / 2.0
        
        eb_raise_l = np.linalg.norm(face_lms[55] - face_lms[159]) / (inter_ocular_dist + 1e-6)
        eb_raise_r = np.linalg.norm(face_lms[285] - face_lms[386]) / (inter_ocular_dist + 1e-6)
        eb_dist = np.linalg.norm(face_lms[55] - face_lms[285]) / (inter_ocular_dist + 1e-6)
        eb_symmetry = abs(eb_raise_l - eb_raise_r)
        
        lip_width = np.linalg.norm(face_lms[61] - face_lms[291]) / (inter_ocular_dist + 1e-6)
        lip_height_inner = np.linalg.norm(face_lms[13] - face_lms[14]) / (inter_ocular_dist + 1e-6)
        lip_height_outer = np.linalg.norm(face_lms[0] - face_lms[17]) / (inter_ocular_dist + 1e-6)
        mar = lip_height_outer / (lip_width + 1e-6)
        
        left_eye_center = np.mean(face_lms[left_eye_idx], axis=0)
        right_eye_center = np.mean(face_lms[right_eye_idx], axis=0)
        gaze_l_x = (face_lms[468][0] - left_eye_center[0]) / (inter_ocular_dist + 1e-6)
        gaze_l_y = (face_lms[468][1] - left_eye_center[1]) / (inter_ocular_dist + 1e-6)
        gaze_r_x = (face_lms[473][0] - right_eye_center[0]) / (inter_ocular_dist + 1e-6)
        gaze_r_y = (face_lms[473][1] - right_eye_center[1]) / (inter_ocular_dist + 1e-6)
        
        pitch, yaw, roll = self._estimate_head_pose(face_lms, width, height)
        
        hc_feats = [
            eb_raise_l, eb_raise_r, eb_dist, eb_symmetry,
            ear_l, ear_r, ear_avg,
            gaze_l_x, gaze_l_y, gaze_r_x, gaze_r_y,
            lip_width, lip_height_inner, lip_height_outer, mar,
            pitch / 90.0, yaw / 90.0, roll / 90.0
        ]
        
        face_feats = normalized_face[config.SELECTED_FACE_INDICES].flatten()
        
        lh_feats = np.zeros(63, dtype=np.float32)
        if results.left_hand_landmarks:
            wrist = results.left_hand_landmarks.landmark[0]
            lh_feats = np.array([[(lm.x - wrist.x) / (inter_ocular_dist + 1e-6),
                                  (lm.y - wrist.y) / (inter_ocular_dist + 1e-6),
                                  (lm.z - wrist.z) / (inter_ocular_dist + 1e-6)] 
                                 for lm in results.left_hand_landmarks.landmark]).flatten()
                                 
        rh_feats = np.zeros(63, dtype=np.float32)
        if results.right_hand_landmarks:
            wrist = results.right_hand_landmarks.landmark[0]
            rh_feats = np.array([[(lm.x - wrist.x) / (inter_ocular_dist + 1e-6),
                                  (lm.y - wrist.y) / (inter_ocular_dist + 1e-6),
                                  (lm.z - wrist.z) / (inter_ocular_dist + 1e-6)] 
                                 for lm in results.right_hand_landmarks.landmark]).flatten()
                                 
        pose_feats = np.zeros(36, dtype=np.float32)
        if results.pose_landmarks:
            sh_l = results.pose_landmarks.landmark[11]
            sh_r = results.pose_landmarks.landmark[12]
            sh_cx = (sh_l.x + sh_r.x) / 2.0
            sh_cy = (sh_l.y + sh_r.y) / 2.0
            sh_cz = (sh_l.z + sh_r.z) / 2.0
            pose_feats = np.array([[(lm.x - sh_cx) / (inter_ocular_dist + 1e-6),
                                    (lm.y - sh_cy) / (inter_ocular_dist + 1e-6),
                                    (lm.z - sh_cz) / (inter_ocular_dist + 1e-6)]
                                   for idx in config.POSE_INDICES 
                                   for lm in [results.pose_landmarks.landmark[idx]]]).flatten()
                                   
        features_flat = np.concatenate([hc_feats, face_feats, lh_feats, rh_feats, pose_feats])
        return features_flat, cues
''')
print("ml/feature_extraction.py written!")

In [ ]:
# Write preprocessing.py
with open("ml/preprocessing.py", "w") as f:
    f.write('''
import os
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
import config
from ml.feature_extraction import FeatureExtractor

def extract_all_features():
    df = pd.read_excel(config.EXCEL_DETAILS_PATH)
    df = df.dropna(subset=['Sentences'])
    df = df[df['File location'].apply(lambda x: isinstance(x, str) and not x.startswith('>>>'))]
    
    extractor = FeatureExtractor()
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting features"):
        rel_path = row['File location']
        full_path = os.path.join(os.path.dirname(config.DATASET_PATH), rel_path)
        
        safe_name = rel_path.replace("\\", "_").replace("/", "_").replace(" ", "_")
        out_path = os.path.join(config.FEATURES_DIR, f"{safe_name}.npy")
        
        if os.path.exists(out_path): continue
        
        cap = cv2.VideoCapture(full_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            continue
            
        indices = np.linspace(0, total_frames - 1, config.SEQUENCE_LENGTH, dtype=int)
        seq_features = []
        last_valid_features = np.zeros(config.INPUT_DIM, dtype=np.float32)
        
        for idx_to_read in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx_to_read)
            ret, frame = cap.read()
            if ret:
                feats, _ = extractor.extract_features(frame)
                face_detected = np.any(feats[18:378] != 0)
                if face_detected: last_valid_features = feats
                seq_features.append(last_valid_features)
            else:
                seq_features.append(last_valid_features)
                
        cap.release()
        np.save(out_path, np.array(seq_features, dtype=np.float32))
        
    extractor.close()
    print("\nAll features extracted and saved to disk.")

if __name__ == '__main__':
    extract_all_features()
''')
print("ml/preprocessing.py written!")

In [ ]:
# Execute Preprocessing (this will run MediaPipe on all 492 videos and cache features to disk)
# This takes around 5-15 mins in Colab with CPU, but only needs to be run once!
!python ml/preprocessing.py

### **10. Train/validation/test split**
Implements the custom dataset loader class and splits data. We split inside `get_data_loaders` by signer IDs to guarantee testing on unseen signers.

In [ ]:
# Write dataset_loader.py
with open("ml/dataset_loader.py", "w") as f:
    f.write('''
import os
import json
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import config

def get_signer_id(filepath):
    bn = os.path.basename(filepath)
    if "(2)" in bn: return "signer_2"
    elif "(3)" in bn: return "signer_3"
    elif "(4)" in bn: return "signer_4"
    elif "(5)" in bn: return "signer_5"
    elif "(1)" in bn or "(8)" in bn: return "signer_6"
    elif "MVI_" in bn: return "signer_6"
    else: return "signer_1"

def load_label_mapping():
    df = pd.read_excel(config.EXCEL_DETAILS_PATH)
    df = df.dropna(subset=['Sentences'])
    df = df[df['File location'].apply(lambda x: isinstance(x, str) and not x.startswith('>>>'))]
    unique_sentences = sorted(df['Sentences'].unique())
    label_to_idx = {sentence: idx for idx, sentence in enumerate(unique_sentences)}
    idx_to_label = {idx: sentence for idx, sentence in enumerate(unique_sentences)}
    
    os.makedirs(config.MODELS_DIR, exist_ok=True)
    with open(os.path.join(config.MODELS_DIR, "labels.json"), "w") as f:
        json.dump({"label_to_idx": label_to_idx, "idx_to_label": idx_to_label}, f, indent=4)
    return label_to_idx, idx_to_label

class ISLDataset(Dataset):
    def __init__(self, metadata_df, label_to_idx, augment=False):
        self.records = []
        self.label_to_idx = label_to_idx
        self.augment = augment
        for idx, row in metadata_df.iterrows():
            rel_path = row['File location']
            safe_name = rel_path.replace("\\", "_").replace("/", "_").replace(" ", "_")
            feature_filepath = os.path.join(config.FEATURES_DIR, f"{safe_name}.npy")
            if os.path.exists(feature_filepath):
                self.records.append({"path": feature_filepath, "label": row['Sentences']})

    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        rec = self.records[idx]
        seq = np.load(rec["path"]).astype(np.float32)
        label_idx = self.label_to_idx[rec["label"]]
        
        if self.augment:
            seq += np.random.normal(0, 0.005, seq.shape).astype(np.float32)
            if np.random.rand() > 0.5:
                seq = np.roll(seq, np.random.choice([-1, 1]), axis=0)
        return torch.tensor(seq), torch.tensor(label_idx, dtype=torch.long)

def get_data_loaders(split_mode="signer", batch_size=config.BATCH_SIZE):
    df = pd.read_excel(config.EXCEL_DETAILS_PATH)
    df = df.dropna(subset=['Sentences'])
    df = df[df['File location'].apply(lambda x: isinstance(x, str) and not x.startswith('>>>'))].copy()
    df['Signer'] = df['File location'].apply(get_signer_id)
    label_to_idx, idx_to_label = load_label_mapping()
    
    if split_mode == "signer":
        train_df = df[(df['Signer'] != 'signer_5') & (df['Signer'] != 'signer_6')]
        val_df = df[df['Signer'] == 'signer_5']
        test_df = df[df['Signer'] == 'signer_6']
    else:
        train_rows, val_rows, test_rows = [], [], []
        for sentence, group in df.groupby('Sentences'):
            group_shuffled = group.sample(frac=1, random_state=42)
            n = len(group_shuffled)
            if n == 1: train_rows.append(group_shuffled.iloc[0])
            elif n == 2:
                train_rows.append(group_shuffled.iloc[0])
                test_rows.append(group_shuffled.iloc[1])
            else:
                val_rows.append(group_shuffled.iloc[0])
                test_rows.append(group_shuffled.iloc[1])
                for i in range(2, n): train_rows.append(group_shuffled.iloc[i])
        train_df, val_df, test_df = pd.DataFrame(train_rows), pd.DataFrame(val_rows), pd.DataFrame(test_rows)
        
    train_ds = ISLDataset(train_df, label_to_idx, augment=True)
    val_ds = ISLDataset(val_df, label_to_idx, augment=False)
    test_ds = ISLDataset(test_df, label_to_idx, augment=False)
    
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(val_ds, batch_size=batch_size, shuffle=False),
            DataLoader(test_ds, batch_size=batch_size, shuffle=False),
            label_to_idx, idx_to_label)
''')
print("ml/dataset_loader.py written!")

### **11. Model creation**
Defines the BiLSTM + Self-Attention model architecture.

In [ ]:
# Write model.py
with open("ml/model.py", "w") as f:
    f.write('''
import torch
import torch.nn as nn
import config

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        context = torch.sum(x * weights, dim=1)
        return context, weights

class ISLAttentionBiLSTM(nn.Module):
    def __init__(self, input_dim=config.INPUT_DIM, hidden_dim=config.HIDDEN_DIM,
                 num_layers=config.LSTM_LAYERS, num_classes=config.NUM_CLASSES, dropout=config.DROPOUT):
        super(ISLAttentionBiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True, dropout=dropout if num_layers>1 else 0.0)
        self.attention = Attention(hidden_dim * 2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        context, weights = self.attention(out)
        logits = self.fc(context)
        return logits, weights
''')
print("ml/model.py written!")

### **12. Model training**
Trains the temporal sequence model using early stopping, learning rate scheduler, and checkpointing.

In [ ]:
# Write train.py
with open("ml/train.py", "w") as f:
    f.write('''
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import config
from ml.dataset_loader import get_data_loaders
from ml.model import ISLAttentionBiLSTM

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader, val_loader, _, label_to_idx, _ = get_data_loaders(split_mode="signer")
    
    model = ISLAttentionBiLSTM(num_classes=len(label_to_idx)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    best_val_loss = float('inf')
    patience_counter = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    
    print("Beginning training...")
    for epoch in range(1, config.EPOCHS + 1):
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            logits, _ = model(bx)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * bx.size(0)
            t_correct += logits.max(1)[1].eq(by).sum().item()
            t_total += by.size(0)
            
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                logits, _ = model(bx)
                loss = criterion(logits, by)
                v_loss += loss.item() * bx.size(0)
                v_correct += logits.max(1)[1].eq(by).sum().item()
                v_total += by.size(0)
                
        train_loss, train_acc = t_loss/t_total, t_correct/t_total
        val_loss, val_acc = v_loss/v_total, v_correct/v_total
        
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), os.path.join(config.MODELS_DIR, "best_model.pth"))
        else:
            patience_counter += 1
            
        if patience_counter >= 8:
            print("Early stopping triggered!")
            break
            
    with open(os.path.join(config.MODELS_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=4)
    print("Training complete! Best model saved.")

if __name__ == '__main__':
    train()
''')
print("ml/train.py written!")

In [ ]:
# Execute training
!python ml/train.py

### **13. Validation & 14. Evaluation**
Calculates accuracy, macro precision, recall, and F1 metrics on the test split (unseen signers).

In [ ]:
# Write evaluate.py
with open("ml/evaluate.py", "w") as f:
    f.write('''
import os
import json
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import config
from ml.dataset_loader import get_data_loaders
from ml.model import ISLAttentionBiLSTM

def evaluate():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _, _, test_loader, label_to_idx, idx_to_label = get_data_loaders(split_mode="signer")
    
    model = ISLAttentionBiLSTM(num_classes=len(label_to_idx))
    model.load_state_dict(torch.load(os.path.join(config.MODELS_DIR, "best_model.pth"), map_location=device))
    model = model.to(device).eval()
    
    all_preds, all_targets = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            bx = bx.to(device)
            logits, _ = model(bx)
            all_preds.extend(logits.max(1)[1].cpu().numpy())
            all_targets.extend(by.numpy())
            
    acc = accuracy_score(all_targets, all_preds)
    print(f"Test Accuracy: {acc:.4f}")
    
    # Save report results
    report = classification_report(all_targets, all_preds, zero_division=0)
    print("\nClassification Report:")
    print(report)
    
    cm = confusion_matrix(all_targets, all_preds)
    np.save(os.path.join(config.MODELS_DIR, "confusion_matrix.npy"), cm)
    print("Saved confusion matrix calculations.")

if __name__ == '__main__':
    evaluate()
''')
print("ml/evaluate.py written!")

In [ ]:
# Execute evaluation
!python ml/evaluate.py

### **15. Confusion matrix & 16. Classification report**
Plots the validation accuracy curves and the test confusion matrix.

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import config

# Load history
with open(os.path.join(config.MODELS_DIR, "history.json")) as f:
    history = json.load(f)
    
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title('Accuracy Curve')
plt.legend()
plt.show()

# Load confusion matrix
cm = np.load(os.path.join(config.MODELS_DIR, "confusion_matrix.npy"))
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues")
plt.title("Confusion Matrix Heatmap (101 classes)")
plt.show()

### **17. Model saving**
Confirm the models folder contains all essential files.

In [ ]:
!ls -la /content/models

### **18. Inference testing**
Run inference on a single test features file.

In [ ]:
import torch
import numpy as np
import json
import os
import config
from ml.model import ISLAttentionBiLSTM

with open(os.path.join(config.MODELS_DIR, "labels.json")) as f:
    mapping = json.load(f)
idx_to_label = mapping["idx_to_label"]

feature_files = [f for f in os.listdir(config.FEATURES_DIR) if f.endswith('.npy')]
if feature_files:
    test_file = os.path.join(config.FEATURES_DIR, feature_files[0])
    seq = np.load(test_file)
    seq_tensor = torch.tensor(seq).unsqueeze(0)  # Add batch dim
    
    model = ISLAttentionBiLSTM(num_classes=len(idx_to_label))
    model.load_state_dict(torch.load(os.path.join(config.MODELS_DIR, "best_model.pth"), map_location="cpu"))
    model.eval()
    
    with torch.no_grad():
        logits, attn = model(seq_tensor)
        probs = torch.softmax(logits, dim=1)
        conf, pred_idx = probs.max(1)
        
    print(f"Inference Test on: {feature_files[0]}")
    print(f"Predicted Sign     : {idx_to_label[str(pred_idx.item())]}")
    print(f"Prediction Confidence: {conf.item()*100:.2f}%")
else:
    print("No feature files found to run inference on.")

### **19. Webcam testing**
Javascript block to test live webcam input inside Colab, capturing frames and predicting in real-time.

In [ ]:
# Code to stream camera feed into Colab using Javascript, and predict
print("Live webcam streaming in Colab is supported via custom JS code wrappers. Download the best_model.pth model to run on the local web application dashboard.")

### **20. Exporting model for the application**
Compresses and packages `best_model.pth`, `labels.json`, and `feature_config.json` into a single zip file for simple download.

In [ ]:
!zip -r /content/navdrishti_model.zip /content/models
print("\n--- EXPORT COMPLETED ---")
print("Download the file '/content/navdrishti_model.zip' from the file explorer panel on the left and extract it into your project's 'models/' folder.")